# Сравнение deepvk vs Qwen2.5-VL на GQA-ru / MMBench-ru (lmms-eval)

Ноутбук для VK Education проекта: сравнение deepvk-модели и Qwen2.5-VL на бенчмарках **GQA-ru** и **MMBench-ru** через фреймворк `lmms-eval`.

**Порядок работы (важно соблюдать по шагам):**
1. Проверить GPU
2. Установить зависимости
3. **Перезапустить runtime** (обязательно — иначе конфликт версий numpy/protobuf)
4. Авторизоваться в Hugging Face
5. Клонировать и установить `lmms-eval`
6. Быстрый sanity-check (`--limit 8`)
7. Полный прогон deepvk-модели
8. Полный прогон Qwen2.5-VL
9. Сборка сравнительной таблицы и график
10. Сохранение результатов (zip + Google Drive)

В комментариях к каждой ячейке — почему сделано именно так: это места, где Colab обычно падает (numpy, flash-attn, accelerate multiprocessing, CUDA OOM).

## 1. Проверка GPU
Если попадётся T4 (~15 GB) — используем Qwen2.5-VL-3B вместо 7B, чтобы не словить CUDA OOM.

In [ ]:
!pip install -q decord

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

import subprocess
gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True
).stdout.strip()

try:
    gpu_mem_mb = int(gpu_info.splitlines()[0])
except Exception:
    gpu_mem_mb = 0

print(f"GPU memory: {gpu_mem_mb} MB")
RECOMMEND_SMALL_QWEN = gpu_mem_mb < 20000  # T4/L4 ~15-16GB -> берём 3B, A100 40GB+ -> 7B ок
print("Рекомендация:", "Qwen2.5-VL-3B-Instruct" if RECOMMEND_SMALL_QWEN else "Qwen2.5-VL-7B-Instruct")


## 2. Установка зависимостей

Что и почему:

- **`numpy<2`** — предустановленный в Colab образ иногда тянет numpy 2.x, который ломает совместимость с torch/opencv из коробки → жёстко пинуем.
- **`transformers>=4.49.0`** — более старые версии не знают класс `Qwen2_5_VLForConditionalGeneration`, попытка загрузить модель упадёт с `KeyError`/`ValueError: Unrecognized model`.
- **Без flash-attn** — на Colab (T4/L4/A100 без preinstalled wheel) сборка flash-attn часто падает или занимает 20+ минут. Вместо этого везде ниже используем `attn_implementation=sdpa` — работает из коробки, без компиляции.
- **Без `.[all]` для lmms-eval** — полный набор extras тянет decord/av и другие video-зависимости, которые часто не собираются в Colab и не нужны для GQA/MMBench (это image-задачи). Ставим `lmms-eval` в режиме `-e .` без лишних extras.
- **Без `accelerate launch`** — в интерактивном Jupyter/Colab `accelerate launch` иногда виснет или падает на форке процессов. Ниже везде используется `python -m lmms_eval` напрямую с `device_map=auto` в `model_args` — для одной GPU этого достаточно.

In [ ]:
%%capture
!pip install -q "numpy<2" --force-reinstall
!pip install -q "transformers>=4.49.0" "accelerate>=0.34.0" "tokenizers>=0.20" \
    "qwen-vl-utils==0.0.8" sentencepiece protobuf pillow einops timm \
    "huggingface_hub[hf_xet]"


## ⚠️ 3. ПЕРЕЗАПУСТИТЕ RUNTIME СЕЙЧАС

`Runtime -> Restart session` (или `Ctrl+M .`).

Обязательно: numpy был переустановлен, и без рестарта процесс держит в памяти старую версию — получите `ImportError` или `ValueError: numpy.dtype size changed`.

После рестарта продолжайте со следующей ячейки (ячейки 1 и 2 выше повторно запускать не нужно, GPU и pip install уже сделаны — но переменную `RECOMMEND_SMALL_QWEN` из ячейки 1 надо будет пересчитать, если ядро перезапущено полностью).

In [ ]:
import numpy, transformers, torch

print("numpy:", numpy.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert transformers.__version__ >= "4.49.0", "transformers слишком старый для Qwen2.5-VL — проверьте установку выше"


## 4. Авторизация в Hugging Face
Некоторые датасеты/веса могут упираться в rate-limit при анонимном доступе — логин снимает это. Токен: huggingface.co/settings/tokens (достаточно `read`).

In [ ]:
from huggingface_hub import login
import getpass

hf_token = getpass.getpass("HF token: ")
login(token=hf_token)


## 5. Клонирование и установка lmms-eval

`gqa-ru` и `mmbench_ru_dev` — встроенные задачи lmms-eval (используют датасеты `deepvk/GQA-ru` и `deepvk/MMBench-ru` с Hugging Face напрямую), отдельно ничего скачивать/регистрировать не нужно.

In [ ]:
%%capture
!git clone --depth 1 https://github.com/EvolvingLMMs-Lab/lmms-eval.git /content/lmms-eval
%cd /content/lmms-eval
!pip install -q -e .
%cd /content


In [ ]:
!python -m lmms_eval --tasks list | grep -i -E "gqa-ru|mmbench_ru" || echo "ВНИМАНИЕ: задачи не найдены — проверьте, что установка lmms-eval выше прошла без ошибок"

In [ ]:
path = "/content/lmms-eval/lmms_eval/models/simple/llava_hf.py"

with open(path, "r") as f:
    content = f.read()

old = '        self._image_processor = AutoProcessor.from_pretrained(pretrained, revision=revision, trust_remote_code=trust_remote_code)'

new = old + '''
        if getattr(self._image_processor, "patch_size", None) is None:
            self._image_processor.patch_size = getattr(self._model.config.vision_config, "patch_size", 14)
        if getattr(self._image_processor, "vision_feature_select_strategy", None) is None:
            self._image_processor.vision_feature_select_strategy = getattr(self._model.config, "vision_feature_select_strategy", "default")'''

assert old in content, "Строка для патча не найдена — возможно, версия lmms-eval отличается, пришли мне содержимое файла"
content = content.replace(old, new)

with open(path, "w") as f:
    f.write(content)

print("Патч применён")

## 6. Настройка моделей и sanity-check на 8 примерах

Сначала — быстрый прогон на 8 сэмплах, чтобы убедиться, что веса/датасет скачиваются и метрика считается, прежде чем запускать полный (более долгий) прогон.

In [ ]:
RECOMMEND_SMALL_QWEN = True  # True для T4/L4 (~15GB), False для A100/40GB+

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct" if RECOMMEND_SMALL_QWEN else "Qwen/Qwen2.5-VL-7B-Instruct"
DEEPVK_MODEL_ID = "deepvk/llava-saiga-8b"
DEEPVK_MODEL_TYPE = "llava_hf"

print("Qwen model:", QWEN_MODEL_ID)
print("deepvk model:", DEEPVK_MODEL_ID, "| type:", DEEPVK_MODEL_TYPE)

In [ ]:
!python -m lmms_eval \
  --model qwen2_5_vl \
  --model_args pretrained={QWEN_MODEL_ID},attn_implementation=sdpa,device_map=auto \
  --tasks gqa-ru \
  --batch_size 1 \
  --limit 8 \
  --output_path ./logs_sanity_qwen/

In [ ]:
!pip install -q "numpy<2" "transformers==4.43.4" "huggingface_hub==0.25.2" "tokenizers==0.19.1" --force-reinstall

In [ ]:
!python -m lmms_eval \
  --model {DEEPVK_MODEL_TYPE} \
  --model_args pretrained={DEEPVK_MODEL_ID},device_map=auto,dtype=bfloat16 \
  --tasks gqa-ru \
  --batch_size 1 \
  --limit 8 \
  --force_simple \
  --output_path ./logs_sanity_deepvk/

Если обе ячейки выше отработали и напечатали таблицу с метрикой (даже низкой — на 8 примерах это нормально) — окружение настроено верно, можно переходить к полному прогону.

**Если упало с CUDA OOM:** уменьшите модель (`Qwen2.5-VL-3B` вместо 7B), либо добавьте в `model_args` `load_in_4bit=True` (требует `bitsandbytes`, тогда добавьте `!pip install -q bitsandbytes` в установку выше).

**Если упало с "Unrecognized model" / KeyError по Qwen2_5_VL:** значит после рестарта runtime подхватилась старая версия `transformers` — заново выполните `!pip show transformers` и при необходимости переустановите.

## 7. Полный прогон: deepvk на GQA-ru и MMBench-ru

In [ ]:
!python -m lmms_eval \
  --model {DEEPVK_MODEL_TYPE} \
  --model_args pretrained={DEEPVK_MODEL_ID},device_map=auto,dtype=bfloat16 \
  --tasks gqa-ru,mmbench_ru_dev \
  --batch_size 1 \
  --force_simple \
  --limit 300 \
  --log_samples \
  --log_samples_suffix deepvk \
  --output_path ./logs/

## 8. Полный прогон: Qwen2.5-VL на GQA-ru и MMBench-ru

In [ ]:
!pip install -q "transformers>=4.49.0" "huggingface_hub[hf_xet]" --force-reinstall

In [ ]:
!python -m lmms_eval \
  --model qwen2_5_vl \
  --model_args pretrained={QWEN_MODEL_ID},attn_implementation=sdpa,device_map=auto \
  --tasks gqa-ru,mmbench_ru_dev \
  --batch_size 1 \
  --limit 300 \
  --log_samples \
  --log_samples_suffix qwen25vl \
  --output_path ./logs/

## 9. Сборка сравнительной таблицы

`lmms-eval` кладёт `results.json` в подпапки `output_path` с timestamp в названии. Ниже — код, который сам находит все такие файлы, парсит метрики и собирает единую таблицу + график.

In [ ]:
import glob
import json as _json
import pandas as pd

def collect_results(root="./logs/"):
    rows = []
    paths = glob.glob(f"{root}**/*_results.json", recursive=True)
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            data = _json.load(f)
        if "results" not in data:
            continue
        for task, metrics in data["results"].items():
            row = {"source_file": path, "task": task}
            row.update({k: v for k, v in metrics.items() if not k.startswith("alias")})
            rows.append(row)
    return pd.DataFrame(rows)

df = collect_results("./logs/")
pd.set_option("display.max_colwidth", None)
df

In [ ]:
# Пример построения сравнительного графика — подставьте реальное имя колонки метрики из вывода выше
# (для gqa-ru обычно 'exact_match,none', для mmbench_ru_dev — 'gpt_eval_score,none' либо похожее)
print("Доступные колонки:", list(df.columns))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

gqa = df[df["task"] == "gqa-ru"].copy()
gqa["model"] = gqa["source_file"].str.extract(r"logs/([^/]+)/")
axes[0].bar(gqa["model"], gqa["exact_match,none"])
axes[0].set_title("GQA-ru (exact_match)")
axes[0].set_ylim(0, 1)

mmb = df[df["task"] == "mmbench_ru_dev"].copy()
mmb["model"] = mmb["source_file"].str.extract(r"logs/([^/]+)/")
axes[1].bar(mmb["model"], mmb["gpt_eval_score,none"])
axes[1].set_title("MMBench-ru (gpt_eval_score)")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig("comparison_chart.png", dpi=150)
plt.show()

## 10. Сохранение результатов
Архивируем логи и таблицу — либо скачиваем, либо копируем в Google Drive для отчёта/приложения к проекту.

In [ ]:
!zip -r results_gqa_mmbench_ru.zip ./logs ./comparison_chart.png 2>/dev/null
df.to_csv("comparison_table.csv", index=False)

import shutil
shutil.move("results_gqa_mmbench_ru.zip", "/kaggle/working/results_gqa_mmbench_ru.zip")
shutil.move("comparison_table.csv", "/kaggle/working/comparison_table.csv")

print("Готово — файлы в /kaggle/working/, смотри вкладку Output справа")

## Приложение: если что-то пошло не так

| Симптом | Причина | Решение |
|---|---|---|
| `ValueError: numpy.dtype size changed` | Не перезапустили runtime после установки numpy<2 | `Runtime -> Restart session`, затем заново выполнить ячейку с проверкой версий |
| `Unrecognized model` / `KeyError: qwen2_5_vl` | Слишком старый `transformers` (подхватился системный, не обновлённый) | `!pip show transformers`, при версии <4.49 — переустановить и снова перезапустить runtime |
| `CUDA out of memory` | 7B модель не влезает в T4/L4 | Использовать `Qwen2.5-VL-3B-Instruct`, либо `load_in_4bit=True` с `bitsandbytes`, либо `batch_size=1` (уже стоит) |
| Установка `flash-attn` висит/падает | Нет готового wheel под GPU/версию CUDA в Colab | Не ставить flash-attn вообще, использовать `attn_implementation=sdpa` (уже сделано в ноутбуке) |
| `accelerate launch` зависает без вывода | Проблемы multiprocessing-форка в Jupyter | Использовать `python -m lmms_eval` напрямую (уже сделано в ноутбуке) |
| Задачи `gqa-ru`/`mmbench_ru_dev` не находятся `--tasks list` | Установка lmms-eval прошла с ошибкой или не из того каталога | Повторить шаг 5, проверить, что `pip install -e .` отработал без ошибок (уберите `%%capture`, чтобы увидеть лог) |
| Долго скачиваются датасеты/веса | Не залогинены в HF, либо медленное HF-соединение | Проверить шаг 4 (login), использовать `huggingface_hub[hf_xet]` (уже в установке) |
